In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import math
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

# ==========================================\
# GOOGLE DRIVE BAĞLANTISI
# ==========================================\
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive bağlandı!")
except ImportError:
    print("Colab ortamında değilsiniz, lokal çalışmaya devam ediliyor.")

# ==========================================\
# AYARLAR (A100 GPU)
# ==========================================\
CONFIG = {
    "csv_train": "/content/drive/MyDrive/Colab Notebooks/multimodal_dataset/train.csv",
    "root_dir": "/content/drive/MyDrive/Colab Notebooks/multimodal_dataset",

    "image_size": 224,
    "batch_size": 32,
    "epochs": 100,
    "timesteps": 1000,
    "lr": 2e-4,
    "target_counts": {"FA": 3, "ICGA": 3, "US": 1},
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

print(f"Kullanılan Cihaz: {CONFIG['device'].upper()}")
if CONFIG['device'] == 'cuda':
    print(f"GPU Modeli: {torch.cuda.get_device_name(0)}")

# ==========================================\
# U-NET MİMARİSİ
# ==========================================\
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2 * in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()

    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))
        time_emb = time_emb[(...,) + (None,) * 2] # [B, C, 1, 1]
        h = h + time_emb
        h = self.bnorm2(self.relu(self.conv2(h)))
        return self.transform(h)

class A100_UNet(nn.Module):
    """A100 için derinleştirilmiş DDPM U-Net"""
    def __init__(self):
        super().__init__()
        image_channels = 3
        down_channels = (64, 128, 256, 512)
        up_channels = (512, 256, 128, 64)
        out_dim = 3
        time_emb_dim = 32

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )

        self.conv0 = nn.Conv2d(image_channels, down_channels[0], 3, padding=1)
        self.downs = nn.ModuleList([Block(down_channels[i], down_channels[i+1], time_emb_dim) for i in range(len(down_channels)-1)])
        self.ups = nn.ModuleList([Block(up_channels[i], up_channels[i+1], time_emb_dim, up=True) for i in range(len(up_channels)-1)])
        self.output = nn.Conv2d(up_channels[-1], out_dim, 1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)
        residual_inputs = []
        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)
        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)
            x = up(x, t)
        return self.output(x)

# ==========================================\
# DIFFUSION SCHEDULER
# ==========================================\
class DDPMScheduler:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.timesteps = timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

    def add_noise(self, x_0, t):
        noise = torch.randn_like(x_0)
        sqrt_alpha_prod = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_prod = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        x_t = sqrt_alpha_prod * x_0 + sqrt_one_minus_alpha_prod * noise
        return x_t, noise

    @torch.no_grad()
    def sample(self, model, num_samples, size):
        model.eval()
        x = torch.randn((num_samples, 3, size, size)).to(self.device)
        for i in tqdm(reversed(range(self.timesteps)), desc="Diffusion Sampling", leave=False):
            t = torch.full((num_samples,), i, device=self.device, dtype=torch.long)
            predicted_noise = model(x, t)
            alpha = self.alphas[i]
            alpha_cumprod = self.alphas_cumprod[i]
            beta = self.betas[i]

            if i > 0:
                noise = torch.randn_like(x)
            else:
                noise = torch.zeros_like(x)

            x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * predicted_noise) + torch.sqrt(beta) * noise
        model.train()
        return x

# ==========================================\
# DATASET VE EĞİTİM
# ==========================================\
class RealModalityDataset(Dataset):
    def __init__(self, df, modality, root_dir, transform):
        self.img_paths = []
        for _, row in df.iterrows():
            patient_id = str(row['name']).zfill(6)
            mod_path = os.path.join(root_dir, 'train', patient_id, modality)
            if not os.path.exists(mod_path):
                mod_path = os.path.join(root_dir, patient_id, modality)

            if os.path.exists(mod_path):
                files = [os.path.join(mod_path, f) for f in os.listdir(mod_path) if f.lower().endswith(('.jpg', '.png'))]
                self.img_paths.extend(files)

        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        return self.transform(img)

def train_and_generate():
    transform = transforms.Compose([
        transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    inv_transform = transforms.Compose([
        transforms.Normalize([-1, -1, -1], [2, 2, 2]),
        transforms.ToPILImage()
    ])

    df = pd.read_csv(CONFIG['csv_train'])
    scheduler = DDPMScheduler(timesteps=CONFIG['timesteps'], device=CONFIG['device'])
    scaler = GradScaler()

    trained_models = {}

    for modality, required_count in CONFIG['target_counts'].items():
        print(f"\n{'='*40}\n[{modality}] İÇİN DDPM EĞİTİMİ BAŞLIYOR\n{'='*40}")

        dataset = RealModalityDataset(df, modality, CONFIG['root_dir'], transform)
        if len(dataset) == 0:
            print(f"[{modality}] için gerçek resim bulunamadı, atlanıyor.")
            continue

        dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=False)
        model = A100_UNet().to(CONFIG['device'])
        optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'])
        criterion = nn.MSELoss()

        # EĞİTİM
        for epoch in range(CONFIG['epochs']):
            model.train()
            epoch_loss = 0
            for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']}", leave=False):
                batch = batch.to(CONFIG['device'])
                t = torch.randint(0, CONFIG['timesteps'], (batch.size(0),), device=CONFIG['device']).long()

                noisy_imgs, noise = scheduler.add_noise(batch, t)

                optimizer.zero_grad()
                with autocast():
                    predicted_noise = model(noisy_imgs, t)
                    loss = criterion(predicted_noise, noise)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                epoch_loss += loss.item()

            if (epoch+1) % 10 == 0:
                print(f"[{modality}] Epoch {epoch+1} | Loss: {epoch_loss/len(dataloader):.4f}")

        trained_models[modality] = model

        # ÜRETİM
        print(f"\n--- [{modality}] Eksik Görüntüler Tamamlanıyor ---")
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{modality} Üretimi"):
            patient_id = str(row['name']).zfill(6)
            patient_path = os.path.join(CONFIG['root_dir'], 'train', patient_id)
            if not os.path.exists(patient_path):
                patient_path = os.path.join(CONFIG['root_dir'], patient_id)
                os.makedirs(patient_path, exist_ok=True)

            mod_path = os.path.join(patient_path, modality)
            os.makedirs(mod_path, exist_ok=True)

            existing_files = [f for f in os.listdir(mod_path) if f.lower().endswith(('.jpg', '.png'))]
            current_count = len(existing_files)
            missing_count = required_count - current_count

            if missing_count > 0:
                generated_tensors = scheduler.sample(model, num_samples=missing_count, size=CONFIG['image_size'])
                for i in range(missing_count):
                    img_tensor = generated_tensors[i].cpu().clamp(-1, 1)
                    img_pil = inv_transform(img_tensor)
                    save_name = f"ddpm_synth_{current_count + i + 1}.jpg"
                    img_pil.save(os.path.join(mod_path, save_name))

    print("\nTüm eksik veriler 224x224 boyutunda A100 DDPM mimarisi ile üretilip diske kaydedildi.")

if __name__ == '__main__':
    train_and_generate()

Mounted at /content/drive
Google Drive bağlandı!
Kullanılan Cihaz: CUDA
GPU Modeli: NVIDIA A100-SXM4-40GB


/tmp/ipykernel_444/2593421199.py:197: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



[FA] İÇİN DDPM EĞİTİMİ BAŞLIYOR


Epoch 1/100:   0%|          | 0/44 [00:00<?, ?it/s]/tmp/ipykernel_444/2593421199.py:225: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[FA] Epoch 10 | Loss: 0.0531


...

[US] Epoch 80 | Loss: 0.0277


[US] Epoch 90 | Loss: 0.0232


[US] Epoch 100 | Loss: 0.0298

--- [US] Eksik Görüntüler Tamamlanıyor ---


Streaming output truncated to the last 5000 lines.
Diffusion Sampling: 324it [00:01, 263.98it/s]
Diffusion Sampling: 351it [00:01, 261.63it/s]
Diffusion Sampling: 378it [00:01, 261.03it/s]
Diffusion Sampling: 405it [00:01, 259.84it/s]
Diffusion Sampling: 431it [00:01, 259.43it/s]
Diffusion Sampling: 457it [00:01, 259.28it/s]
Diffusion Sampling: 483it [00:01, 259.26it/s]
Diffusion Sampling: 509it [00:01, 258.72it/s]
Diffusion Sampling: 536it [00:02, 260.50it/s]
Diffusion Sampling: 563it [00:02, 260.36it/s]
Diffusion Sampling: 590it [00:02, 260.00it/s]
Diffusion Sampling: 617it [00:02, 254.35it/s]
Diffusion Sampling: 643it [00:02, 249.79it/s]
Diffusion Sampling: 669it [00:02, 251.85it/s]
Diffusion Sampling: 696it [00:02, 254.79it/s]
Diffusion Sampling: 723it [00:02, 257.20it/s]
Diffusion Sampling: 750it [00:02, 258.25it/s]
Diffusion Sampling: 777it [00:02, 259.00it/s]
Diffusion Sampling: 804it [00:03, 259.53it/s]
Diffusion Sampling: 831it [00:03, 260.50it/s]
Diffusion Sampling: 858it [00


Tüm eksik veriler 224x224 boyutunda A100 DDPM mimarisi ile üretilip diske kaydedildi.


In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, precision_score, recall_score
from kan import KAN

# ==========================================\
# GOOGLE DRIVE BAĞLANTISI
# ==========================================\
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive bağlandı!")
except ImportError:
    print("Colab ortamında değilsiniz, lokal çalışmaya devam ediliyor.")

# ==========================================\
# AYARLAR VE SABİTLEME
# ==========================================\
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random Seed {seed} olarak kilitlendi.")

CONFIG = {
    "csv_train": "/content/drive/MyDrive/Colab Notebooks/multimodal_dataset/train.csv",
    "csv_test": "/content/drive/MyDrive/Colab Notebooks/multimodal_dataset/test.csv",
    "root_dir": "/content/drive/MyDrive/Colab Notebooks/multimodal_dataset",
    "image_size": 224,
    "batch_size": 16,
    "num_epochs": 15,
    "learning_rate": 1e-4,
    "k_folds": 5,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_dir": "/content/drive/MyDrive/Colab Notebooks/experiments/efficientnet_kan_diffusion"
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)
seed_everything()

LABEL_MAP = {
    'Choroidal Hemangioma': 0,
    'Choroidal Melanoma': 1,
    'Choroidal Metastatic Carcinoma': 2
}

# ==========================================\
# DATASET
# ==========================================\
class DiffusionFilledDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.data_frame = dataframe.copy()
        self.data_frame['name'] = self.data_frame['name'].astype(str).apply(lambda x: x.zfill(6))
        self.data_frame['label'] = self.data_frame['pathology'].map(LABEL_MAP)
        self.root_dir = root_dir
        self.transform = transform
        self.modalities = [('FA', 3), ('ICGA', 3), ('US', 1)] # Toplam 7 Kesit

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        row = self.data_frame.iloc[idx]
        patient_id = row['name']
        label = int(row['label'])

        possible_roots = [
            self.root_dir,
            os.path.join(self.root_dir, 'train'),
            os.path.join(self.root_dir, 'test')
        ]

        patient_path = None
        for r in possible_roots:
            candidate = os.path.join(r, patient_id)
            if os.path.exists(candidate):
                patient_path = candidate
                break

        loaded_tensors = []

        if patient_path is not None:
            for mod_name, count in self.modalities:
                mod_path = os.path.join(patient_path, mod_name)
                current_mod_tensors = []

                if os.path.exists(mod_path):
                    files = sorted([os.path.join(mod_path, f) for f in os.listdir(mod_path) if f.lower().endswith(('.jpg', '.png'))])[:count]

                    for img_path in files:
                        try:
                            img = Image.open(img_path).convert('RGB')
                            if self.transform:
                                img = self.transform(img)
                            current_mod_tensors.append(img)
                        except:
                            current_mod_tensors.append(torch.zeros(3, CONFIG['image_size'], CONFIG['image_size']))

                while len(current_mod_tensors) < count:
                    current_mod_tensors.append(torch.zeros(3, CONFIG['image_size'], CONFIG['image_size']))

                loaded_tensors.extend(current_mod_tensors)
        else:
            loaded_tensors = [torch.zeros(3, CONFIG['image_size'], CONFIG['image_size']) for _ in range(7)]

        return torch.stack(loaded_tensors), torch.tensor(label, dtype=torch.long)

transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================\
# HybridEfficientKAN
# ==========================================\
class HybridEfficientKAN(nn.Module):
    def __init__(self, num_classes=3):
        super(HybridEfficientKAN, self).__init__()

        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.backbone.classifier = nn.Identity()

        self.feature_dim = 1280 * 7 # 7 kesit * 1280
        self.projection_dim = 64
        self.projection = nn.Linear(self.feature_dim, self.projection_dim)
        self.relu = nn.ReLU()

        # KAN Sınıflandırıcı
        self.kan = KAN(width=[self.projection_dim, 32, num_classes], grid=5, k=3)
        self.kan.speed()

    def forward(self, x):
        b, num_imgs, c, h, w = x.size()
        x = x.view(b * num_imgs, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, -1)
        projected = self.relu(self.projection(feats))
        out = self.kan(projected)
        return out

# ==========================================\
# 4. ÇAPRAZ DOĞRULAMA
# ==========================================\
def run_evaluation():
    print("Veriler yükleniyor...")
    df_train = pd.read_csv(CONFIG['csv_train'])
    df_test = pd.read_csv(CONFIG['csv_test'])
    df_full = pd.concat([df_train, df_test], ignore_index=True)
    df_full = df_full[df_full['pathology'].isin(LABEL_MAP.keys())].reset_index(drop=True)

    skf = StratifiedKFold(n_splits=CONFIG['k_folds'], shuffle=True, random_state=42)
    labels = df_full['pathology'].map(LABEL_MAP).values

    all_true_labels = []
    all_pred_labels = []

    print(f"{CONFIG['k_folds']}-Fold Başlıyor...\n")

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"--- FOLD {fold+1}/{CONFIG['k_folds']} ---")

        train_sub = df_full.iloc[train_idx]
        val_sub = df_full.iloc[val_idx]

        train_dataset = DiffusionFilledDataset(train_sub, CONFIG['root_dir'], transform=transform)
        val_dataset = DiffusionFilledDataset(val_sub, CONFIG['root_dir'], transform=transform)

        train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

        model = HybridEfficientKAN(num_classes=3).to(CONFIG['device'])
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-4)

        # EĞİTİM
        for epoch in range(CONFIG['num_epochs']):
            model.train()
            train_loss = 0
            for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
                imgs, lbls = imgs.to(CONFIG['device']), lbls.to(CONFIG['device'])
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

        # DOĞRULAMA
        model.eval()
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(CONFIG['device']), lbls.to(CONFIG['device'])
                outputs = model(imgs)
                _, preds = torch.max(outputs, 1)

                all_true_labels.extend(lbls.cpu().numpy())
                all_pred_labels.extend(preds.cpu().numpy())

    # ==========================================\
    # METRİK HESAPLAMA VE RAPORLAMA
    # ==========================================\
    acc = accuracy_score(all_true_labels, all_pred_labels)
    precision = precision_score(all_true_labels, all_pred_labels, average='macro', zero_division=0)
    recall = recall_score(all_true_labels, all_pred_labels, average='macro', zero_division=0)
    f1 = f1_score(all_true_labels, all_pred_labels, average='macro', zero_division=0)

    cm = confusion_matrix(all_true_labels, all_pred_labels)

    # Specificity Hesaplama
    specificities = []
    for i in range(len(LABEL_MAP)):
        tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
        fp = np.sum(cm[:, i]) - cm[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(spec)
    macro_spec = np.mean(specificities)

    print("\n" + "="*40)
    print(" DIFFUSION & NO TTA SONUÇLARI ")
    print("="*40)
    print(f"Accuracy: %{acc*100:.2f}")
    print(f"Macro Precision: %{precision*100:.2f}")
    print(f"Macro Recall (Sensitivity): %{recall*100:.2f}")
    print(f"Macro Specificity: %{macro_spec*100:.2f}")
    print(f"Macro F1-Score: %{f1*100:.2f}")

    # Karmaşıklık Matrisi
    class_names = list(LABEL_MAP.keys())
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 16, "weight": "bold"})
    plt.xlabel('', fontsize=14, weight='bold')
    plt.ylabel('', fontsize=14, weight='bold')
    plt.title('', fontsize=16, weight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['save_dir'], 'matrix_diffusion_no_tta.png'), dpi=300)
    plt.close()

    pd.DataFrame({'True_Label': all_true_labels, 'Pred_Label': all_pred_labels}).to_csv(os.path.join(CONFIG['save_dir'], 'preds_diffusion_no_tta.csv'), index=False)
    print(f"\nAnaliz ve matris görseli {CONFIG['save_dir']} klasörüne kaydedildi.")

if __name__ == '__main__':
    run_evaluation()

ModuleNotFoundError: No module named 'kan'